# 04. Exploratory Data Analysis

## Objective

This notebook describes customer and purchasing behavior before feature engineering. Target analysis is restricted to the development period. The final test remains untouched. One snapshot represents one customer at one monthly reference date, while one purchase event represents one non-cancellation invoice.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

snapshots = pd.read_parquet(
    "../data/interim/labeled_customer_snapshots.parquet"
)
purchase_events = pd.read_parquet(
    "../data/interim/purchase_invoice_events.parquet"
)
cancellation_events = pd.read_parquet(
    "../data/interim/cancellation_invoice_events.parquet"
)

development_snapshots = snapshots.loc[
    snapshots["FinalSplit"].eq("development")
].copy()
development_active = development_snapshots.loc[
    ~development_snapshots["IsChurnedAtReference"]
].copy()
development_end = development_snapshots["ReferenceDate"].max() + pd.offsets.MonthBegin(1)
development_purchases = purchase_events.loc[
    purchase_events["PurchaseDate"].lt(development_end)
].copy()

print("Development snapshots:", development_snapshots.shape)
print("Active modeling snapshots:", development_active.shape)
print("Development purchase invoices:", development_purchases.shape)

## 1. Customer population over time

`Customers` counts everyone with at least one purchase that remains valid at the reference date. `ActiveCustomers` have not yet reached 60 days of inactivity. `ChurnedCustomers` have already crossed that boundary. `NewlyObservedCustomers` made their first observed valid purchase during the preceding calendar month. This is observed acquisition inside the dataset, not necessarily the customer's true first purchase with the retailer.

In [ ]:
development_snapshots["IsNewlyObserved"] = (
    development_snapshots["FirstObservedPurchaseDate"].dt.to_period("M")
    == (
        development_snapshots["ReferenceDate"] - pd.offsets.MonthBegin(1)
    ).dt.to_period("M")
)
population_by_reference = (
    development_snapshots.groupby("ReferenceDate")
    .agg(
        Customers=("Customer ID", "size"),
        ChurnedCustomers=("IsChurnedAtReference", "sum"),
        NewlyObservedCustomers=("IsNewlyObserved", "sum"),
    )
)
population_by_reference["ActiveCustomers"] = (
    population_by_reference["Customers"]
    - population_by_reference["ChurnedCustomers"]
)
population_by_reference["NewCustomerShareOfActive"] = (
    population_by_reference["NewlyObservedCustomers"]
    / population_by_reference["ActiveCustomers"] * 100
)
population_by_reference[[
    "Customers", "ActiveCustomers", "ChurnedCustomers",
    "NewlyObservedCustomers", "NewCustomerShareOfActive"
]].round(2)

In [ ]:
plt.figure(figsize=(11, 5))
plt.plot(population_by_reference.index, population_by_reference["ActiveCustomers"], marker="o", label="Active")
plt.plot(population_by_reference.index, population_by_reference["ChurnedCustomers"], marker="o", label="Already churned")
plt.title("Observed customer population at each reference date")
plt.xlabel("Reference date")
plt.ylabel("Customers")
plt.legend()
plt.tight_layout()
plt.show()

## 2. Churn target over time

The denominator contains only customers active at the reference date. `ChurnRateNext30Days` is the proportion that will reach 60 consecutive days without a valid purchase during the following 30 days. Variation across months matters because a random split would mix these temporal regimes.

In [ ]:
target_by_reference = (
    development_active.groupby("ReferenceDate")["WillChurnNext30Days"]
    .agg(ActiveSnapshots="size", ChurnsNext30Days="sum")
)
target_by_reference["ChurnRateNext30Days"] = (
    target_by_reference["ChurnsNext30Days"]
    / target_by_reference["ActiveSnapshots"] * 100
)
target_by_reference.round(2)

In [ ]:
plt.figure(figsize=(11, 5))
plt.plot(
    target_by_reference.index,
    target_by_reference["ChurnRateNext30Days"],
    marker="o",
)
plt.axhline(
    development_active["WillChurnNext30Days"].mean() * 100,
    color="grey", linestyle="--", label="Development average",
)
plt.title("Churn during the next 30 days")
plt.xlabel("Reference date")
plt.ylabel("Churn rate (%)")
plt.legend()
plt.tight_layout()
plt.show()

## 3. Monthly purchasing activity

Monthly buyers are separated into customers first observed in that month and customers already observed earlier. The table describes acquisition and repeat purchasing without introducing a reactivation target. A single annual cycle can suggest seasonality, but it cannot establish a stable seasonal pattern.

In [ ]:
development_purchases["PurchaseMonth"] = (
    development_purchases["PurchaseDate"].dt.to_period("M").dt.to_timestamp()
)
first_purchase_month = (
    purchase_events.groupby("Customer ID")["PurchaseDate"].min()
    .dt.to_period("M").dt.to_timestamp()
)
development_purchases = development_purchases.join(
    first_purchase_month.rename("FirstPurchaseMonth"),
    on="Customer ID",
)
development_purchases["IsNewBuyer"] = (
    development_purchases["PurchaseMonth"]
    == development_purchases["FirstPurchaseMonth"]
)

monthly_activity = (
    development_purchases.groupby("PurchaseMonth")
    .agg(
        MonthlyPurchasingCustomers=("Customer ID", "nunique"),
        PositiveInvoices=("Invoice", "nunique"),
    )
)
new_buyers = (
    development_purchases.loc[development_purchases["IsNewBuyer"]]
    .groupby("PurchaseMonth")["Customer ID"].nunique()
)
monthly_activity["NewlyObservedCustomers"] = new_buyers.reindex(
    monthly_activity.index, fill_value=0
)
monthly_activity["ExistingPurchasingCustomers"] = (
    monthly_activity["MonthlyPurchasingCustomers"]
    - monthly_activity["NewlyObservedCustomers"]
)
monthly_activity["NewBuyerShare"] = (
    monthly_activity["NewlyObservedCustomers"]
    / monthly_activity["MonthlyPurchasingCustomers"] * 100
)
monthly_activity.round(2)

In [ ]:
plt.figure(figsize=(11, 5))
plt.plot(monthly_activity.index, monthly_activity["NewlyObservedCustomers"], marker="o", label="Newly observed buyers")
plt.plot(monthly_activity.index, monthly_activity["ExistingPurchasingCustomers"], marker="o", label="Existing buyers")
plt.title("Monthly purchasing customers")
plt.xlabel("Purchase month")
plt.ylabel("Customers")
plt.legend()
plt.tight_layout()
plt.show()

## 4. Purchase rhythm known at each snapshot

For every active development snapshot, historical purchases are filtered exactly as they would be during scoring. The median interval is calculated only when the customer has at least two valid historical invoices. The stacked chart shows the share of these repeat-purchase customers in each interval range at every reference date.

In [ ]:
snapshot_purchase_history = (
    development_active[["Customer ID", "ReferenceDate"]]
    .merge(purchase_events, on="Customer ID")
)
snapshot_purchase_history = snapshot_purchase_history.loc[
    snapshot_purchase_history["PurchaseDate"].lt(
        snapshot_purchase_history["ReferenceDate"]
    )
    & (
        snapshot_purchase_history["FullCancellationDate"].isna()
        | snapshot_purchase_history["FullCancellationDate"].gt(
            snapshot_purchase_history["ReferenceDate"]
        )
    )
]
snapshot_purchase_history = snapshot_purchase_history.sort_values(
    ["Customer ID", "ReferenceDate", "PurchaseDate"]
)
snapshot_purchase_history["GapDays"] = (
    snapshot_purchase_history.groupby(
        ["Customer ID", "ReferenceDate"]
    )["PurchaseDate"].diff().dt.total_seconds() / 86400
)
snapshot_rhythm = (
    snapshot_purchase_history.dropna(subset=["GapDays"])
    .groupby(["Customer ID", "ReferenceDate"], as_index=False)
    .agg(
        GapCount=("GapDays", "count"),
        MedianGapDays=("GapDays", "median"),
        MeanGapDays=("GapDays", "mean"),
        GapStdDays=("GapDays", "std"),
    )
)
snapshot_rhythm["MedianGapRange"] = pd.cut(
    snapshot_rhythm["MedianGapDays"],
    bins=[-np.inf, 15, 30, 60, np.inf],
    labels=["0-15 days", "16-30 days", "31-60 days", "More than 60 days"],
)
rhythm_distribution = (
    pd.crosstab(
        snapshot_rhythm["ReferenceDate"],
        snapshot_rhythm["MedianGapRange"],
        normalize="index",
    ) * 100
)
rhythm_distribution.round(2)

In [ ]:
ax = rhythm_distribution.plot(
    kind="bar", stacked=True, figsize=(12, 6), colormap="Greens"
)
ax.set_title("Historical median time between purchases")
ax.set_xlabel("Reference date")
ax.set_ylabel("Repeat-purchase customers (%)")
ax.set_ylim(0, 100)
ax.legend(title="Median interval", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## 5. Invoice-recording time

The following figures count invoice events by recorded hour and weekday. These timestamps may reflect order processing or business operating hours rather than the exact time when the customer decided to buy. They should not automatically become behavioral features.

In [ ]:
purchases_by_hour = (
    development_purchases["PurchaseDate"].dt.hour.value_counts().sort_index()
)
plt.figure(figsize=(10, 4))
plt.bar(purchases_by_hour.index, purchases_by_hour.values)
plt.title("Positive invoices by recorded hour")
plt.xlabel("Hour of day")
plt.ylabel("Invoices")
plt.xticks(range(24))
plt.tight_layout()
plt.show()

In [ ]:
weekday_order = [
    "Monday", "Tuesday", "Wednesday", "Thursday",
    "Friday", "Saturday", "Sunday",
]
purchases_by_weekday = (
    development_purchases["PurchaseDate"].dt.day_name()
    .value_counts().reindex(weekday_order, fill_value=0)
)
plt.figure(figsize=(10, 4))
plt.bar(purchases_by_weekday.index, purchases_by_weekday.values, color="tab:orange")
plt.title("Positive invoices by recorded weekday")
plt.xlabel("Weekday")
plt.ylabel("Invoices")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

## 6. Unusual invoice sizes

The IQR rule flags observations for inspection; it does not prove that they are errors. Purchase variables are strongly right-skewed, which is compatible with a mix of ordinary and wholesale customers. The logarithmic x-axis makes the central box and extreme invoices visible together without deleting data.

In [ ]:
invoice_variables = [
    "InvoiceValue", "InvoiceQuantity", "ProductLines", "UniqueProducts"
]
positive_size_invoices = development_purchases.loc[
    development_purchases["InvoiceValue"].gt(0)
    & development_purchases["InvoiceQuantity"].gt(0)
].copy()
outlier_rows = []
positive_size_invoices["IsInvoiceOutlier"] = False
for feature in invoice_variables:
    q1 = positive_size_invoices[feature].quantile(0.25)
    q3 = positive_size_invoices[feature].quantile(0.75)
    upper_limit = q3 + 1.5 * (q3 - q1)
    feature_outlier = positive_size_invoices[feature].gt(upper_limit)
    positive_size_invoices["IsInvoiceOutlier"] |= feature_outlier
    outlier_rows.append({
        "Feature": feature,
        "Q1": q1,
        "Q3": q3,
        "UpperIQRLimit": upper_limit,
        "FlaggedInvoices": feature_outlier.sum(),
        "FlaggedShare": feature_outlier.mean() * 100,
    })
outlier_summary = pd.DataFrame(outlier_rows).set_index("Feature")
outlier_summary.round(2)

In [ ]:
for feature in invoice_variables:
    plt.figure(figsize=(11, 2.5))
    plt.boxplot(positive_size_invoices[feature], vert=False)
    plt.xscale("log")
    plt.title(f"{feature} on a logarithmic scale")
    plt.xlabel("Observed value, log scale")
    plt.yticks([])
    plt.tight_layout()
    plt.show()

### 6.1 Recurring large-order customers

A large invoice is defined descriptively as being in the top 1% of positive development invoice values. Customers with several such invoices provide evidence that at least part of the extreme tail reflects recurring high-value behavior rather than isolated data corruption. The threshold is not yet a modeling feature and must not be learned from the final test.

In [ ]:
large_invoice_threshold = positive_size_invoices["InvoiceValue"].quantile(0.99)
large_invoices = positive_size_invoices.loc[
    positive_size_invoices["InvoiceValue"].ge(large_invoice_threshold)
]
large_customer_summary = (
    large_invoices.groupby("Customer ID")
    .agg(
        LargeInvoices=("Invoice", "nunique"),
        TotalLargeInvoiceValue=("InvoiceValue", "sum"),
        MaximumInvoiceValue=("InvoiceValue", "max"),
    )
    .sort_values(["LargeInvoices", "TotalLargeInvoiceValue"], ascending=False)
)
print("Top 1% threshold:", round(large_invoice_threshold, 2))
print("Large invoices:", len(large_invoices))
print("Customers concerned:", len(large_customer_summary))
print("Customers with repeated large invoices:", (large_customer_summary["LargeInvoices"] > 1).sum())
large_customer_summary.head(15)

## 7. Geographic context

The latest valid historical invoice supplies the country known at each active snapshot. Because the United Kingdom dominates the data, the stable first comparison is `UK` versus `Non-UK`. Country is descriptive context, not proof of a causal geographic effect.

In [ ]:
latest_country = (
    snapshot_purchase_history.sort_values("PurchaseDate")
    .groupby(["Customer ID", "ReferenceDate"], as_index=False)
    .tail(1)[["Customer ID", "ReferenceDate", "Country"]]
)
geographic_snapshots = development_active.merge(
    latest_country, on=["Customer ID", "ReferenceDate"], validate="one_to_one"
)
geographic_snapshots["CustomerRegion"] = np.where(
    geographic_snapshots["Country"].eq("United Kingdom"),
    "UK", "Non-UK",
)
geography_summary = (
    geographic_snapshots.groupby("CustomerRegion")
    .agg(
        Snapshots=("Customer ID", "size"),
        Customers=("Customer ID", "nunique"),
        MedianRecencyDays=("RecencyDays", "median"),
        ChurnRateNext30Days=("WillChurnNext30Days", "mean"),
    )
)
geography_summary["ChurnRateNext30Days"] *= 100
geography_summary.round(2)

## 8. EDA summary

### Findings carried into feature engineering

- The development population contains 27,594 active snapshots and an overall 31.67% next-30-day churn rate. Monthly rates range from 24.62% to 52.13%, so temporal validation is mandatory.
- Purchasing volume rises strongly before the end-of-year period and reaches its development maximum in November 2010. The dataset contains only one complete annual cycle, so month-related variables may help prediction but do not establish a causal seasonal effect.
- Across reference dates, about 31% of repeat-purchase customers have a historical median interval of 31 to 60 days and about 32% exceed 60 days. Absolute recency should therefore be complemented by recency relative to each customer's historical purchase rhythm.
- About 79% of positive invoices are recorded between 10:00 and 15:00. Recorded hour and weekday mostly describe the retailer's invoice-processing pattern and are not automatically reliable customer-intent features.
- Invoice size variables are strongly right-skewed. Depending on the variable, the IQR rule flags about 5% to 8% of positive invoices. The top-1% value threshold identifies 77 customers, including 40 with repeated large invoices. These are inspection signals, not deletion rules.
- Full cancellations must remain point-in-time events. Only cancellations known before a reference date may affect its features.
- Next-30-day churn rates are 31.63% for UK snapshots and 32.08% for non-UK snapshots. Geography therefore has almost no standalone separation and should remain a low-priority candidate.

### Limitations

- Activity before December 2009 is unavailable, so customer tenure is left-censored.
- Repeated monthly snapshots from one customer are correlated observations.
- One annual cycle cannot establish stable seasonality.
- Invoice timestamps may reflect internal processing rather than the customer's decision time.
- Full-reversal matching is conservative and depends on exact customer, product, quantity, value, and a 30-day matching window.

### Next step

Notebook 5 will create point-in-time RFM features first. Each new feature family will be evaluated before another is added.